# Experiment

## Import libraries

In [ ]:
import pandas as pd

file_path = "mobilization_vietstock_2000_2025.csv"

df = pd.read_csv(file_path)


# Set indicator names as lowercase with underscores
# df["Chỉ tiêu"] = (
#     df["Chỉ tiêu"].str.lower().str.replace(",", "").str.replace(" ", "_")
# )

column_name = "Chỉ tiêu"

df[column_name] = (
    df[column_name]
    .str.lower()
    .str.replace(
        r"[^a-z0-9_\s-]", "", regex=True
    )  # remove everything except letters, numbers, underscore, space
    .str.replace(r"[\s-]+", "_", regex=True)  # replace any whitespace with underscore
)

id_vars = ["Chỉ tiêu", "Đơn vị tính"]

# Melt from wide to long format
df = df.melt(
    id_vars=id_vars,
    var_name="month_str",
    value_name="value",
)

# Clean numeric values
df["value"] = df["value"].astype(str).str.replace(",", "", regex=False)
df["value"] = pd.to_numeric(df["value"], errors="coerce")

# Extract year and month
df["date"] = pd.to_datetime(df["month_str"], errors="coerce")

# Drop rows where date couldn't be parsed
df = df.dropna(subset=["date"])

# Extract numeric year, month
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year

# Use pivot_table with first() to handle duplicates
df = df.pivot_table(
    index=["year", "month"],
    columns=id_vars[0],
    values="value",
    aggfunc="first",
).reset_index()

# Sort by year and month
df = df.sort_values(["year", "month"]).reset_index(drop=True)

# Fill missing values with 0
df.fillna(0, inplace=True)

df

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_22884\833672169.py:38: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(df["month_str"], errors="coerce")


Chỉ tiêu,year,month,deposits_from_economic_organizations,deposits_from_residents,total_payment_instruments
0,2012,4,1084405.00,1449453.00,2533858.00
1,2012,5,1107567.00,1504045.00,2611612.00
2,2012,6,1143442.00,1519428.00,2662870.00
3,2012,7,1163303.00,1530364.00,2693667.00
4,2012,8,1171974.00,1555889.00,2727863.00
...,...,...,...,...,...
153,2025,1,7433824.82,7188027.38,14621852.20
154,2025,2,7362206.43,7366165.80,14728372.23
155,2025,3,7520233.24,7469971.36,14990204.60
156,2025,4,7625146.09,7537620.83,15162766.92


In [4]:
df.columns

Index(['year', 'month', 'deposits_from_economic_organizations',
       'deposits_from_residents', 'total_payment_instruments'],
      dtype='object', name='Chỉ tiêu')